# OrganicAI — 협력 층위(L0~L4) 판정 파이프라인시나리오 하나를 넣으면 두 에이전트의 협력 층위를 판정한다.**처음 쓴다면 `docs/SCENARIO_TEST.md`를 먼저 읽을 것.** (API 키 발급 · Colab 설정 · 에러 대처)## 실행 순서 요약| 단계 | 언제 ||---|---|| 0. 설치 | 세션 최초 1회 || 1~3 | 매 세션 필수 (순서 지킬 것) || 4~5 | 시나리오를 새로 썼을 때 || 6 | 데이터 수집 |> ⚠️ **런타임을 재시작했다면 0번을 건너뛰고 1번부터 다시 실행한다.**> `!git pull` 로 코드를 받은 뒤에도 **반드시 런타임을 재시작**해야 새 코드가 반영된다.

## 0. 설치 (세션 최초 1회)

In [ ]:
!git clone -b yr https://github.com/ewha-oi/OrganicAI.git%cd OrganicAI!pip install -r requirements.txt -qimport syssys.path.append('src')print("Python:", sys.version)

## 1. 세션 시작**런타임 재시작 후에는 여기서부터 실행한다.**

In [ ]:
%cd /content/OrganicAIimport syssys.path.append('src')!git log --oneline -1

In [ ]:
from google.colab import userdataAPI_KEYS = {    "gemini": userdata.get('GEMINI_API_KEY'),   # 현재 미사용 - '없음'이어도 정상    "groq":   userdata.get('GROQ_API_KEY'),     # 필수}for k, v in API_KEYS.items():    print(f"{k:8s} {'OK' if v else '!! 없음 - Secrets 이름/노트북 액세스 토글 확인'}")

In [ ]:
# 채점자(judge) 설정.# 반드시 coop_pipeline 을 임포트하는 어떤 셀보다 먼저 실행할 것 —# llm.py 가 임포트 시점에 이 값을 한 번만 읽는다.# 값을 바꾸려면 여기서 고치고 런타임을 재시작해야 반영된다.import osos.environ["COOP_JUDGE_PROVIDER"] = "groq"                 # anthropic / groq / geminios.environ["COOP_JUDGE_MODEL"]    = "openai/gpt-oss-120b"  # 부록의 모델 목록 참고print("provider =", os.environ["COOP_JUDGE_PROVIDER"])print("model    =", os.environ["COOP_JUDGE_MODEL"])

## 2. 환경 점검 (API 호출 없음, 무료)여기서 걸리는 문제는 실행해도 똑같이 걸린다. 먼저 통과시킬 것.

In [ ]:
!python -m pytest tests/ -q!python tools/dryrun_frame.py

In [ ]:
# 시나리오 형식 + 현재 모델 구성 확인from coop_pipeline.runner import check_scenario_dirfrom coop_pipeline.llm import MODELS, judge_provider, judge_model, judge_key_namecheck_scenario_dir("scenarios")print(f"\nalpha : {MODELS['alpha']}")print(f"beta  : {MODELS['beta']}")print(f"judge : {judge_provider()}:{judge_model()}   (필요한 키: {judge_key_name()})")

## 3. [임시] alpha 모델 대체Google이 Gemini API 접근을 차단해서(전 모델 403/404) alpha를 Groq 모델로 돌린다.**Gemini가 복구되면 이 셀만 실행하지 않으면 원래 설계로 돌아간다. 코드 수정 불필요.**그때는 `COOP_ALPHA_MODEL` 환경변수로 모델 ID만 지정하면 된다.> 이 대체 때문에 alpha·beta·judge가 모두 Groq 무료 티어를 쓴다.> 분당 토큰 한도(TPM)에 걸리기 쉬우므로 `max_turns` 를 6 이하로 두는 것이 안전하다.

In [ ]:
import re, typesfrom groq import Groqfrom coop_pipeline import agents, llmfrom coop_pipeline.runner import load_scenarioGROQ_KEY    = API_KEYS["groq"]ALPHA_MODEL = "qwen/qwen3.6-27b"_THINK = re.compile(r"<think>.*?</think>\s*", re.S)# 이 모델이 어떤 추론 옵션을 받는지 먼저 확인한다 (모델마다 다르고, 틀리면 400)EXTRA = {}for cand in ({"reasoning_effort": "low", "reasoning_format": "hidden"},             {"reasoning_format": "hidden"}, {"reasoning_effort": "low"}, {}):    try:        Groq(api_key=GROQ_KEY).chat.completions.create(            model=ALPHA_MODEL, messages=[{"role": "user", "content": "ping"}],            max_tokens=64, **cand)        EXTRA = cand        break    except Exception as e:        print("불가:", cand, "|", str(e)[:90])print("사용할 옵션:", EXTRA, "\n")class _Resp:    def __init__(self, text): self.text = textclass _GroqModel:    def __init__(self, model_id): self.model_id = model_id    def generate_content(self, prompt):        r = Groq(api_key=GROQ_KEY).chat.completions.create(            model=self.model_id,            messages=[{"role": "user", "content": prompt}],            temperature=agents.TEMPERATURE,            max_tokens=4096,          # 없으면 추론 토큰에 다 쓰고 빈 응답이 온다            **EXTRA,        )        text = _THINK.sub("", r.choices[0].message.content or "").strip()        if not text:            raise RuntimeError(f"alpha({self.model_id}) 빈 응답 — max_tokens 부족")        return _Resp(text)_shim = types.SimpleNamespace(configure=lambda **kw: None, GenerativeModel=_GroqModel)agents._gemini_model = lambda: _shimllm.MODELS["alpha"] = ALPHA_MODELagents.RATE_LIMIT_SLEEP = 3.0# 스모크 테스트는 반드시 '실제 길이의' 프롬프트로 한다.# 짧은 ping은 통과해도 실제 프롬프트에서 토큰 한도에 걸리는 일이 있다.sc = load_scenario("scenarios/A4/A4_simple_energy_campaign.json")v = sc["task_variants"]task = v.get("alpha") or v["shared"]      # 비대칭/대칭 시나리오 양쪽 모두 대응long_prompt = agents.SYSTEM_PROMPT_TEMPLATES["명시"].format(    name="alpha", partner="beta", task=task) + "\n\n지금까지의 대화:\n(없음)\n\n너의 다음 발언:"out = _GroqModel(ALPHA_MODEL).generate_content(long_prompt).textprint(f"길이 {len(out)}자")print(out[:300])print("\nMODELS:", llm.MODELS)

## 4. 발화 태깅 점검채점자가 발화 코드(phatic/meta/lead/arch/agree/comp)를 제대로 붙이는지 5개 사례로 본다.**4/5 이상이면 통과.** 3/5 이하면 채점자 모델을 의심할 것.

In [ ]:
from coop_pipeline.llm import make_judgefrom coop_pipeline.tagging import tag_turnCASES = [    ("수요일 B실로 하자.",     "좋아, 그렇게 하자.",                                 "phatic"),    ("수요일에 하는 게 어때?",  "맞네, 나는 A실을 생각했는데 수요일이면 B실이 맞겠다.",  "agree"),    ("수요일 B실로 하자.",     "좋아. 그런데 예산 확인도 필요해 보여.",                "comp"),    ("예산은 200이야.",       "응, 200이지.",                                     "phatic"),    ("회의 준비 시작하자.",    "지금 정할 건 요일이야. 시간은 나중에.",                "lead"),]j = make_judge(API_KEYS["groq"])hit = 0for prev, cur, want in CASES:    got = tag_turn(j, [{"turn": 1, "speaker": "alpha", "text": prev}],                   {"turn": 2, "speaker": "beta", "text": cur})    ok = (got["codes"] == ["phatic"]) if want == "phatic" else (want in got["codes"])    hit += ok    print(f"{'O' if ok else 'X'} 기대={want:6s} 실제={got['codes']} ref={got['ref']}")    print(f"   근거: {got['evidence']}")print(f"\n{hit}/5")

## 5. 시나리오 실기동**새 시나리오를 썼다면 여기서 한 번 돌려본다.**통과 기준은 하나뿐이다 — **에러 없이 `L0`~`L4` 중 하나가 나오는 것.**어느 층위가 나오든, `[FAIL] Q3` 이 뜨든 상관없다. 그건 실험 결과이지 형식 오류가 아니다.

In [ ]:
# 시나리오 경로만 본인 파일로 바꿔서 쓴다.from coop_pipeline.runner import run_scenarioresult = run_scenario(    "scenarios/A4/A4_simple_energy_campaign.json",    condition="명시",     # 또는 "묵시"    api_keys=API_KEYS,    n_solo=2,            # 시험용. 정식 실행은 5    max_turns=6,         # 시험용. 정식 실행은 10 (단, 임시 alpha는 TPM 한도 주의)    out_dir="runs_pilot",)

In [ ]:
# 무엇이 실제로 일어났는지 들여다보기 (에러 원인 추적용)log = result["log"]print("=== 1) 대화가 실제로 오갔는가 ===")for t in log["turns"]:    print(f"[{t['turn']}] {t['speaker']}: {t['text'][:150]}")print("\n=== 2) 태깅이 붙었는가 ===")for t in log["turns"]:    print(t["turn"], t["speaker"], t["codes"], "ref:", t["ref"])print("\n=== 3) 무엇을 채점했는가 ===")print(log["group_output_text"][:500])print("\n단독:", log.get("solo_grades") or log.get("solo_scores"),      "/ 그룹:", log.get("group_grade") or log.get("group_score"))

In [ ]:
# A1 경로 점검 — 비대칭 지문(alpha/beta) + 체크리스트 채점 + 퍼센타일 판정.# A2(shared)와는 다른 코드 경로를 타므로 별도로 한 번 확인한다.from coop_pipeline.runner import run_scenariores_A1 = run_scenario(    "scenarios/A1/A1_simple_meeting.json",    condition="명시",    api_keys=API_KEYS,    n_solo=2,    max_turns=6,    out_dir="runs_pilot",)print("\n판정 :", res_A1["level"])print("병목 :", res_A1["stopped_at"] or "없음 (L4까지 통과)")

In [ ]:
# 저장된 로그를 다시 판정한다. API 호출 0, 무료.!ls -R runs_pilotfrom coop_pipeline.runner import classify_saved_dir_ = classify_saved_dir("runs_pilot")

## 6. 정식 실행명시/묵시 두 조건을 돌린다. 단독 산출물은 한 번만 만들어 두 조건이 공유하므로조건 간 비교의 기준선이 통일된다.> ⚠️ 임시 alpha(Groq 무료 티어)로 `max_turns=10` 을 쓰면 분당 토큰 한도에 걸릴 수 있다.> `413 Request too large` 가 뜨면 `max_turns` 를 낮출 것.> 대화 원본은 `runs/raw/` 에 태깅 전에 저장되므로 뒤 단계에서 실패해도 유실되지 않는다.

In [ ]:
from coop_pipeline.runner import run_scenario_both_conditionsresults = run_scenario_both_conditions(    "scenarios/A4/A4_simple_energy_campaign.json",    api_keys=API_KEYS, n_solo=5, max_turns=10, out_dir="runs",)print(results["명시"]["level"], results["묵시"]["level"])

In [ ]:
# Drive 백업 — 실행이 끝나면 바로 할 것. Colab 세션이 끊기면 runs/ 는 사라진다.from google.colab import drivedrive.mount('/content/drive')!mkdir -p /content/drive/MyDrive/OrganicAI_runs!cp -r runs/* /content/drive/MyDrive/OrganicAI_runs/!ls /content/drive/MyDrive/OrganicAI_runs/

In [ ]:
# 임계값을 바꿔가며 판정이 어떻게 달라지는지 본다. API 호출 0, 무료.## 주의: 결과를 보고 나서 결과에 맞춰 기준을 고치면 안 된다.#       여기서는 '어느 기준이 결과를 좌우하는지' 관찰만 하고,#       확정은 데이터를 충분히 모은 뒤 configs/thresholds_v2.json 으로 한다.from coop_pipeline.runner import classify_saved_dirfrom coop_pipeline import load_thresholdsfor gap in (2, 1.5, 1, 0.5):    print(f"\n### grade_gap_min = {gap}")    classify_saved_dir("runs", dict(load_thresholds("v1"), grade_gap_min=gap))

## 부록

In [ ]:
# 최신 코드 받아오기. 받은 뒤에는 반드시 런타임을 재시작하고 1번부터 다시 실행할 것.!git pull!git log --oneline -3

In [ ]:
# Groq에서 지금 쓸 수 있는 모델 목록.# 모델이 퇴역해 실행이 통째로 실패할 때 여기서 대체 ID를 고른다.# 규칙: judge 는 alpha·beta 어느 쪽과도 다른 계열이어야 한다 (self-preference 편향).from groq import Groqfor m in sorted(x.id for x in Groq(api_key=API_KEYS["groq"]).models.list().data):    print("  ", m)